## **Goal**
**Given the ground truth (Unreal Engine) coordinates and camera rotation vector, calculate:**
* Pointing vector (shoulder joint -> wrist joint) in camera coordinates
* Distance from shoulder joint to camera
* Angles between shoulder-wrist and shoulder-camera vectors

### **Setup**

```
conda create -n fbv-gt-calc python=3.10
conda activate fbv-gt-calc
conda install numpy
conda install ipykernel
```

In [1]:
import numpy as np

### **Given**

In [2]:
# Keypoint Global Coordinates

shoulder_coords_global = np.array([-4581.538, -185.471, 141.476])
wrist_coords_global = np.array([-4528.306, -185.768, 141.553])
camera_coords_global = np.array([-4079.628, -319.957, 441.476])
camera_forward_global = np.array([-0.837, 0.224, -0.500])

In [3]:
# Assumed Constants

camera_coords = np.array([0, 0, 0])
world_up = np.array([0, 0, 1])

In [4]:
# Expected Results

expected_distance = 600.0 # cm
expected_yaw = -15.0      # deg
expected_pitch = 30.0     # deg
expected_angular_separation = np.sqrt(expected_yaw**2 + expected_pitch**2)

### **Calculations**

In [5]:
# Create Rotation Matrix

camera_right_global = np.cross(camera_forward_global, world_up)
camera_right_global /= np.linalg.norm(camera_right_global)

camera_up_global = np.cross(camera_forward_global, camera_right_global)
camera_up_global /= np.linalg.norm(camera_up_global)

rotation_matrix = np.vstack((camera_forward_global,
                             camera_right_global,
                             camera_up_global))

print(rotation_matrix)

[[-0.837       0.224      -0.5       ]
 [ 0.25852455  0.96600469 -0.        ]
 [ 0.48282253 -0.12921415 -0.86613285]]


In [6]:
# Translate To Origin

shoulder_coords = shoulder_coords_global - camera_coords_global
wrist_coords = wrist_coords_global - camera_coords_global

print(shoulder_coords)
print(wrist_coords)

[-501.91   134.486 -300.   ]
[-448.678  134.189 -299.923]


In [7]:
# Apply Rotation Transformation

shoulder_coords = rotation_matrix @ shoulder_coords
wrist_coords = rotation_matrix @ wrist_coords

print(shoulder_coords)
print(wrist_coords)

[6.00223534e+02 1.58048523e-01 1.28906115e-01]
[555.563322    13.63292408  25.80219918]


### **Analysis**

In [8]:
# Shoulder-Camera Distance

distance = np.linalg.norm(shoulder_coords)
print(f"Distance should be close to {expected_distance} cm:")
print(distance, "cm")

Distance should be close to 600.0 cm:
600.2235686505245 cm


In [9]:
# Shoulder-Wrist Shoulder-Camera Vectors

shoulder_wrist = wrist_coords - shoulder_coords
shoulder_wrist /= np.linalg.norm(shoulder_wrist)

shoulder_camera = camera_coords - shoulder_coords
shoulder_camera /= np.linalg.norm(shoulder_camera)

print(shoulder_wrist)
print(shoulder_camera)

[-0.83873938  0.25306438  0.48215628]
[-9.99999942e-01 -2.63316090e-04 -2.14763502e-04]


In [10]:
# Angular Separation

angular_separation_rad = np.arccos(np.clip(np.dot(shoulder_wrist, shoulder_camera), -1.0, 1.0))
angular_separation_deg = np.rad2deg(angular_separation_rad)

print(f"Angular separation should be close to {expected_angular_separation} deg:")
print(angular_separation_deg)


Angular separation should be close to 33.54101966249684 deg:
33.010668454817996


### **Angular Separation Components**
Given the angular separation of the shoulder-wrist and shoulder-camera vectors, and camera pitch angle (which could be
retreived from an onboard gyroscope), calculate the vertical component (pitch/elevation) and the horizontal component
(yaw/azimuth) of the angular separation in global coordinates.

In [11]:
# Camera Pitch (Ground Truth / Gyroscope)

camera_pitch_rad = -np.asin(camera_forward_global[-1])
camera_pitch_deg = np.rad2deg(camera_pitch_rad)
print(f"Camera pitch should be close to {expected_pitch} deg:")
print(camera_pitch_deg, "deg")

Camera pitch should be close to 30.0 deg:
30.000000000000004 deg


In [12]:
# Camera Un-Pitch Matrix (Aligns Z-Axis with Gravity)

unpitch_matrix = np.array([[np.cos(-camera_pitch_rad), 0, np.sin(-camera_pitch_rad)],
                         [0, 1, 0],
                         [-np.sin(-camera_pitch_rad), 0, np.cos(-camera_pitch_rad)]])

print(unpitch_matrix)

[[ 0.8660254  0.        -0.5      ]
 [ 0.         1.         0.       ]
 [ 0.5        0.         0.8660254]]


In [13]:
# Un-Pitch Vectors

shoulder_wrist_gravity = unpitch_matrix @ shoulder_wrist
shoulder_wrist_gravity /= np.linalg.norm(shoulder_wrist_gravity)

shoulder_camera_gravity = unpitch_matrix @ shoulder_camera
shoulder_camera_gravity /= np.linalg.norm(shoulder_camera_gravity)

print(shoulder_wrist_gravity)
print(shoulder_camera_gravity)

[-0.96744775  0.25306438 -0.0018101 ]
[-8.65917972e-01 -2.63316090e-04 -5.00185962e-01]


In [14]:
# Yaw & Pitch Components

shoulder_wrist_gravity_yaw = np.rad2deg(np.atan2(shoulder_wrist_gravity[1], shoulder_wrist_gravity[0]))
shoulder_wrist_gravity_pitch = np.rad2deg(np.asin(np.clip(shoulder_wrist_gravity[-1], -1.0, 1.0)))

shoulder_camera_gravity_yaw = np.rad2deg(np.atan2(shoulder_camera_gravity[1], shoulder_camera_gravity[0]))
shoulder_camera_gravity_pitch = np.rad2deg(np.asin(np.clip(shoulder_camera_gravity[-1], -1.0, 1.0)))

delta_yaw = (shoulder_wrist_gravity_yaw - shoulder_camera_gravity_yaw + 180) % 360 - 180
delta_pitch = shoulder_wrist_gravity_pitch - shoulder_camera_gravity_pitch

print(delta_yaw)
print(delta_pitch)

-14.676368164693486
29.908592833694883
